# OCR-Anchored Grounding and CART Failure Analysis

**Name:** John-Ronan Beira  
**Date:** 21 November 2026  
**Dataset:** ScreenSpot-Pro, Windows subset (Li et al., 2025), 1,020 grounding requests

Using all 1,020 ScreenSpot-Pro Windows requests, this study evaluates a text-anchored grounding baseline and uses CART to examine conditions associated with failure.

The baseline is narrower than a full framework. It does not capture screens, inject input,
run actions, close the loop, or use a vision model. Jev remains future work.


## 1. Research question and scope

**Research question.** How accurately can an OCR ensemble ground English instructions to
annotated targets in high-resolution Windows screenshots, and how well can request-level
conditions predict grounding failure on applications held out during validation?

**Scope**

- The evaluation covers one baseline adapter and harness, not a model-agnostic framework.
- No resolution effect: the EDA already showed that resolution and application are
  confounded (15 of 17 applications were captured at one resolution).
- No Jev evaluation: Jev is the future decision-layer adapter, not a component here.
- No end-to-end computer use: there is no execution or verification.


## 2. Method

1. **OCR ensemble.** RapidOCR and Tesseract (PSM 3 and 11) produce text boxes.
  Tesseract recognizes some English labels more accurately; RapidOCR performs better on
  small text and Chinese characters.
2. **Bilingual matcher.** Each candidate is scored against the English instruction and its
   supplied Chinese translation. English uses token coverage, token F1, fuzzy similarity
   and token containment; Chinese uses character fuzzy similarity and character overlap.
3. **Point-in-box evaluation.** The centre of the selected box is the prediction. It is
   `correct` if inside the ground-truth box, `mislocalized` if outside, and `not_found` if
   the matcher rejects every candidate.
4. **CART failure model.** Features are known before the target box is revealed:
   application, target type, resolution, instruction length, OCR coverage and match
   confidence. Validation is grouped by application.

The main results use all 1,020 requests, preserving the corpus mix: 622 text and 398 icon
requests. A 300-request balanced pilot is available for comparison, though its class mix
differs from the corpus. The setup cell loads `records_all.jsonl` when present; otherwise,
it uses `records_300.jsonl`.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from isdm.dm import cart, features, plots, summary
from isdm.workspace.paths import repo_root

ROOT = repo_root()
DATA_DIR = ROOT / "data" / "screenspot_pro_windows"
RECORDS = ROOT / "runs" / "grounding" / "records_all.jsonl"
if not RECORDS.is_file():
    RECORDS = ROOT / "runs" / "grounding" / "records_300.jsonl"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("records:", RECORDS)
print("exists :", RECORDS.is_file())


## 3. Data and baseline outcomes

Generate the full 1,020-request run with:

```bash
uv run python scripts/run_grounding.py --n 1020 \
  --output runs/grounding/records_all.jsonl
```

Each request is written as a separate JSON object, so partial runs can still be inspected.

Run the 300-request balanced pilot with:

```bash
uv run python scripts/run_grounding.py --per-type 150 \
  --output runs/grounding/records_300.jsonl
```

The setup cell loads the full-run file when it exists and otherwise uses the pilot.


In [ ]:
if not RECORDS.is_file():
    raise FileNotFoundError(
        "Run `uv run python scripts/run_grounding.py --per-type 150` first."
    )

records = features.load_records(RECORDS)
print("records:", len(records))
display(
    records[
        ["id", "ui_type", "instruction", "label", "candidate_text", "score", "distance_px"]
    ].head(10)
)


### What the baseline did

The tables report icon and text targets separately. The baseline can match icons only when
they have an OCR-readable label; without one, this method cannot locate the icon. Text
targets are where an OCR-based approach is most likely to succeed.


In [ ]:
display(summary.label_counts(records))
display(summary.accuracy_by(records, "ui_type"))
display(summary.accuracy_by(records, "application"))
print("overall accuracy:", f"{summary.overall_accuracy(records):.1%}")


In [ ]:
print(f"corpus-weighted strict accuracy: {summary.corpus_weighted_accuracy(records):.1%}")
for tolerance in (25, 50, 100):
    print(f"within {tolerance} px: {summary.relaxed_accuracy(records, tolerance):.1%}")
    display(summary.relaxed_accuracy_by(records, "ui_type", tolerance))
    weighted = summary.corpus_weighted_accuracy(records, tolerance)
    print(f"corpus-weighted within {tolerance} px: {weighted:.1%}")


In [ ]:
plots.plot_label_counts(records)
plt.show()

plots.plot_accuracy_by_application(records)
plt.show()

plots.plot_error_distance(records)
plt.show()


## 4. CART failure model

The CART analysis uses application, resolution, instruction length, OCR coverage and match
confidence, along with `ui_type` from the benchmark annotations. Validation keeps each
application in one fold. A deployed system would not know `ui_type` before grounding; the
text-only model is closer to that setting.


In [ ]:
frame, categorical, numeric = features.model_frame(records)
print("rows:", len(frame), "failure rate:", f"{frame['failure'].mean():.1%}")
print("categorical:", categorical)
print("numeric:", numeric)
display(frame[["ui_type", "application", "score", "ocr_boxes", "failure"]].head())


In [ ]:
all_result = cart.evaluate_cart(
    frame,
    categorical,
    numeric,
    max_depth=4,
    min_samples_leaf=5,
    n_splits=5,
    class_weight=None,
)
display(all_result.fold_metrics)
display(all_result.summary.round(3))
display(all_result.importance.head(15))
print(all_result.rules)


In [ ]:
cart.plot_cart(all_result, max_depth=3)
plt.show()

plots.plot_importance(all_result)
plt.show()


### Text-only CART

Most icon requests fail because the baseline has no visual detector. Among text targets,
we examine which features predict failure when OCR can read the target.



In [ ]:
text_frame = features.text_only(frame)
print("text rows:", len(text_frame), "failure rate:", f"{text_frame['failure'].mean():.1%}")
text_result = cart.evaluate_cart(
    text_frame,
    categorical,
    numeric,
    max_depth=4,
    min_samples_leaf=15,
    n_splits=5,
)
display(text_result.fold_metrics)
display(text_result.summary.round(3))
display(text_result.importance.head(15))
print(text_result.rules)


In [ ]:
display(summary.model_summary(("all targets", all_result), ("text only", text_result)))

In [ ]:
plots.plot_importance(text_result)
plt.show()

cart.plot_cart(text_result, max_depth=3)
plt.show()


### Post-hoc geometry model

This model also uses the annotated target box to add its area and position. Those
features are unavailable before grounding, so this model describes conditions associated
with failure; it cannot predict failure in a live system.


In [ ]:
geo_frame, geo_categorical, geo_numeric = features.model_frame(
    records, include_geometry=True
)
geo_result = cart.evaluate_cart(
    geo_frame,
    geo_categorical,
    geo_numeric,
    max_depth=4,
    min_samples_leaf=20,
    n_splits=5,
)
display(geo_result.fold_metrics)
display(geo_result.summary.round(3))
display(geo_result.importance.head(15))
print(geo_result.rules)


In [ ]:
display(
    summary.model_summary(
        ("all targets", all_result),
        ("text only", text_result),
        ("geometry", geo_result),
    )
)

## 5. Discussion

Strict accuracy was 29.9% for text targets and 1.8% for icon targets. RapidOCR and Tesseract
with bilingual matching recovered some cases missed by either OCR engine alone, including
English instructions paired with Chinese UI labels. In the all-target CART, matcher score was
the strongest feature, followed by target type. The text-only tree relied mainly on matcher
score, score margin and instruction length; without a visual detector, the baseline cannot
locate icons that lack readable labels.

The analysis keeps each application's requests in the same validation fold because requests
are nested within applications and 14 screens are duplicated. We do not estimate a resolution
effect: 15 of 17 applications were captured at a single resolution, so resolution and
application are confounded in this dataset.

## 6. Limitations

- The main analysis uses all 1,020 requests. The 300-request balanced pilot is only a
  secondary comparison, not the source of the reported results.
- The baseline cannot identify icons without a readable text label or recover text that
  neither OCR engine recognizes.
- The all-target model uses `ui_type`, which is available in the benchmark annotations
  but unavailable to a deployed system; the text-only model is closer to that setting.
- The geometry model uses the ground-truth box, so its results are explanatory rather
  than predictive.
- Jev integration, screen capture, action execution and verification remain future work.

## 7. Next steps

1. Compare the full-corpus results with the 300-request balanced pilot.
2. If access becomes available, test a Jev candidate-selector adapter on a 100–200 request
   subset; the harness interface can stay as it is.
3. Add an icon-aware candidate source only if it can be evaluated without leaking the
   ground-truth box.
4. Report per-application and per-target-type breakdowns in the paper, not a single
   aggregate accuracy.

## 8. References

- Li, K., Meng, Z., Lin, H., Luo, Z., Tian, Y., Ma, J., Huang, Z., & Chua, T.-S. (2025).
  *ScreenSpot-Pro: GUI Grounding for Professional High-Resolution Computer Use.*
  arXiv:2504.07981.
- Dataset: `likaixin/ScreenSpot-Pro`, Hugging Face.
- RapidOCR (ONNX Runtime) and Tesseract OCR.
- scikit-learn CART (`DecisionTreeClassifier`), grouped cross-validation.
